In [ ]:
import io
import os
import re
import csv
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from huggingface_hub import hf_hub_download
from datasets import DatasetDict, Image, load_dataset

# OpenFake dataset

## Downloading a shard of the dataset

I downloaded a single shard of the training data to open and inspect.

See https://huggingface.co/datasets/ComplexDataLab/OpenFake

In [ ]:
train_shard0 = pd.read_parquet("../data/train-00000-of-00032-00000.parquet", engine="pyarrow")

In [ ]:
train_shard0

Loading the parquet file into a dataframe, we can see there are 6 columns and 4000 rows, with each row corresponding to an image.

The columns are:
- image (bytes)

- prompt: the prompt used to generate the image, or for real images this is just a caption
- label: 'real' or 'fake'
- model: for fake images this is the generator name e.g 'flux.2-dev', and for real images this is the name of the dataset the image comes from e.g 'laion'
- type: generator class: base, finetune, lora, image (for non-generator real photos), video (for frames extracted from text-to-video / image-to-video models).
- release_date: first release date of the generator, or collection date for real images. Format varies (YYYY-MM or YYYY-MM-DD).

In [ ]:
# Data correction: all 227 'sd-2.1' rows are tagged release_date "2024-01", but
# Stable Diffusion 2.1 was actually released in December 2022.
train_shard0.loc[train_shard0["model"] == "sd-2.1", "release_date"] = "2022-12"

### Feature distributions

In [ ]:
train_shard0["label"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

# Fixed, colorblind-safe categorical order (never cycled/rainbow).
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
OTHER_COLOR = "#9c9c9c"


def plot_pie(counts: pd.Series, title: str, top_n: int = 7) -> None:
    """Pie chart from a value_counts series, folding anything past top_n into 'Other'.

    Category names go in a legend rather than on the wedges, so labels for
    similarly-sized slices don't overlap; only the percentage stays on the wedge.
    """
    if len(counts) > top_n:
        counts = pd.concat([counts.iloc[:top_n], pd.Series({"Other": counts.iloc[top_n:].sum()})])
    colors = CATEGORICAL[: len(counts)]
    if "Other" in counts.index:
        colors[-1] = OTHER_COLOR
    plt.figure(figsize=(6, 5))
    wedges, _, _ = plt.pie(counts, autopct="%1.0f%%", colors=colors, startangle=90, pctdistance=0.8)
    plt.legend(wedges, counts.index, loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
train_shard0["label"].value_counts()

In [ ]:
train_shard0[train_shard0["label"] == "fake"]["model"].value_counts()

In [ ]:
plot_pie(train_shard0["label"].value_counts(), "Label distribution")

We have roughly 2000 images in each of the 'fake' and 'real' subsets.

In [ ]:
counts = train_shard0.loc[train_shard0["label"] == 'real', "model"].value_counts()
plot_pie(counts[counts > 1], f"Model distribution (real)", top_n = 20)

The real images come from the 'laion' and 'pexels' dataset, nearly 50/50.

The huggingface dataset source states that 'laion' contains 'politically salient or newsworthy images' and 'carries authentic web compression artifacts'.

The 'pexels' dataset is described as 'high-quality stock photographs'.

In [ ]:
fake = train_shard0[train_shard0["label"] == "fake"]
fake_counts = fake["model"].value_counts()
fake_counts = fake_counts[fake_counts > 10].sort_values()

# A few models have a handful of rows tagged with a stray off-year; take the mode
# so each model gets a single representative release year.
model_year = fake.groupby("model")["release_date"].agg(lambda s: s.str[:4].mode()[0])
labels = [f"{model} ({model_year[model]})" for model in fake_counts.index]

In [ ]:
fake_type = train_shard0.loc[train_shard0["label"] == "fake", "type"].replace(
    {"Base": "base"}
)
plot_pie(fake_type.value_counts(), "Type distribution (fake)")

68% of the fake images come from base models, which presumably overlaps with the 'frontier proprietary models' and 'open source flagships' categories listed on the huggingface dataset website. The other 32%, which include 'finetune' and 'lora', presumably overlap with the 'community fine-tunes and LoRAs (sampled from Civitai)' category.

In [ ]:
# Same models/counts as the bar chart above, but year is a real axis instead of label text.
from adjustText import adjust_text

plt.figure(figsize=(9, 7))
years = model_year.loc[fake_counts.index].astype(int)
plt.scatter(years, fake_counts.values, color=CATEGORICAL[0])
texts = [
    plt.annotate(model, (year, count), fontsize=8)
    for model, year, count in zip(fake_counts.index, years, fake_counts.values)
]
adjust_text(texts, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))
plt.xlabel("Release year")
plt.ylabel("Count")
plt.title("Model usage vs. release year (fake, count > 20)")
plt.tight_layout()
plt.show()

This scatter plot shows that the majority of the fake images were generated by models released in 2024, e.g stable diffusion 3.5 and flux.1-dev. The former is the most significant contributor, with ~250 images, followed by stable diffusion 2.1, with ~225, which was released in 2022. The majority of models contribute between 20 and 70 fake images to the dataset.

For the 12 generators contributing at least 50 images to the dataset, I asked Claude to look up their architecture, e.g GAN, diffusion or LLM-based. 11 of the models cam back as diffusion-based, and only one as LLM-based (GPT image-1, contributing only 60 images).

To get more LLM-generated images I'd need to download more shards of the dataset, but there's a risk that the distribution of generator types in each shard will be similar. 

A solution to this could be streaming; see a later section! Before that, let's take a closer look at the diffusion-generated images downloaded in this shard.

### Diffusion-generated images

## Streaming the dataset

Instead of downloading shards of the dataset only to find that they contain very few non-diffusion-generated images, by streaming the huggingface dataset I should be able to filter on the model names that I'm interested and download images from only those models.

In [ ]:
dataset = load_dataset("ComplexDataLab/OpenFake", "core", split="train", streaming=True)

In [ ]:
print(next(iter(dataset)))

In [ ]:
# this is a list of detectors with an LLM component

mllm_generators = {
    "gpt-image-1",
    "gpt-image-1.5",
    "gpt-image-2.0",
    "nano-banana",
    "nano-banana-pro",
    "imagen-3",
    "imagen-4",
    "ideogram-2.0",
    "ideogram-3.0",
    "grok-2-image-1212",
    "DALL·E 3",
    "qwen-image",
    "z-image-turbo",
    "mystic",
    "aurora",
    "lumina",
    "kolors",
    "hidream-i1-full",
}

In [ ]:
# streams the huggingface dataset and iterates through each row
# if the model is in the mllm_generators list, then the image gets saved locally
# the metadata for each saved image, e.g. filename, model, label, release_date, is written to a manifest.csv file

MAX_IMAGES_PER_GENERATOR = 50

OUTPUT_FOLDER = "../data/mllm_fakes"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

ds = load_dataset("ComplexDataLab/OpenFake", "core", split="train", streaming=True)
ds = ds.cast_column(
    "image", Image(decode=False)
)  # now row["image"] is {"bytes":..., "path":...}, otherwise when I load a row I get an error when the image is not a valid image file

kept = Counter()

with open(f"{OUTPUT_FOLDER}/manifest.csv", "w", newline="") as manifest:
    w = csv.writer(manifest)
    w.writerow(["filename", "model", "label", "release_date"])

    for row_num, row in tqdm(enumerate(ds)):
        if row["label"] != "fake":
            continue
        model = row["model"]
        if model not in mllm_generators or kept[model] >= MAX_IMAGES_PER_GENERATOR:
            continue
        filename = f"{model}_{kept[model]:04d}.png"  # PNG, not JPG — see below
        try:
            img = PILImage.open(io.BytesIO(row["image"]["bytes"]))
            img.save(f"{OUTPUT_FOLDER}/{filename}")
        except Exception as e:
            print(f"  skipped bad image ({model}): {e!r}")
            continue  # don't count it, don't write manifest row, move on

        w.writerow([filename, model, row["label"], row["release_date"]])
        kept[model] += 1

        if all(kept[m] >= MAX_IMAGES_PER_GENERATOR for m in mllm_generators):
            break

print(kept)

In [ ]:
folder_path = Path(OUTPUT_FOLDER)
len(os.listdir(folder_path))

We have 280 images generated by MLLMs.

Let's look at how many images come from each model.

In [ ]:
pattern = re.compile(r"^(.+)_\d{4}\.\w+$")
counts = Counter(
    m.group(1)
    for p in Path("../data/mllm_fakes").iterdir()
    if (m := pattern.match(p.name))
)
labels, values = zip(*counts.most_common())

colors = [
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
    "#4a3aa7",
    "#e34948",
][: len(labels)]

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    values,
    labels=labels,
    colors=colors,
    autopct=lambda p: f"{round(p / 100 * sum(values))}",
    pctdistance=0.75,
    startangle=90,
    counterclock=False,
)
ax.set_title("Distribution of MLLM-generated images by model")
plt.tight_layout()
plt.show()

Here are the release dates of these models:

qwen-image: 2025-08-02
nano-banana: 2025-08
mystic: 2024-01
ideogram-3.0: 2025-03
hidream-i1-full: 2024-01
grok-2-image-1212: 2024-12
gpt-image-1: 2025-04